In [11]:

import akshare as ak
import pymysql

import pandas as pd;
from pandas import DataFrame

In [3]:
!python3 --version

Python 3.10.13


In [8]:
!pip3 install --upgrade pip

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.6 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pip
    Found existing installation: pip 23.3.1
    Uninstalling pip-23.3.1:
      Successfully uninstalled pip-23.3.1


In [10]:
!pip3 install akshare

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [7]:
!python3 -m pip upgrade

ERROR: unknown command "upgrade"


In [1]:
!pip3 install mysql-connector-python

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [7]:
#defs:

#puting together:

import akshare as ak
import pymysql

import pandas as pd;
from pandas import DataFrame

import pymysql
from prettyprinter import pprint
from sqlalchemy import create_engine
import traceback

remote_sql_server = '8.210.152.214'
password = 'Zhibin1982'

engine = create_engine("mysql+mysqlconnector://root:" + password + "@" + remote_sql_server + ":3306/stock?charset=utf8")
#data.to_sql("test_table", engine, index=False)

def get_dateframe_result_ex(sql, debug=True):
    db = pymysql.connect(
    host = remote_sql_server , # 服务器地址
    port = 3306, # 端口
    user = "root" , # 用户名
    passwd = password , # 密码
    db = 'stock')
    cursor = db.cursor()
    
    
    try:
        cursor.execute(sql)
    except:
        print("sql exception when executing:" + sql)
        return False, DataFrame();
        
        
    #result = cursor.fetchall()
    #fields = cursor.description
    

    data = cursor.fetchall()
    columnDes = cursor.description #获取连接对象的描述信息
    if debug:
        print(columnDes);
    columnNames = [columnDes[i][0] for i in range(len(columnDes))] #获取列名
    df = pd.DataFrame([list(i) for i in data],columns=columnNames)
     
    #sql_data = pd.DataFrame (cursor.fetchall()) 
    #sql_data.columns = columnNames;
        
    cursor.close()
    db.close()
    if debug:
      pprint(df)
    return True,df;

def change_col_to_varchar(table_name,colname, debug=True):
    db = pymysql.connect(
    host = remote_sql_server , # 服务器地址
    port = 3306, # 端口
    user = "root" , # 用户名
    passwd = password , # 密码
    db = 'stock')
    # 创建游标
    cur = db.cursor()

#create table sys_data1 like sys_data;
#create index idx_value on sys_data1(value(28));
#INSERT into sys_data1 SELECT * from sys_data;
#drop table sys_data;
#rename table sys_data1 to sys_data;
 
#原文链接：https://blog.csdn.net/KeepLearnZhangXiaoBo/article/details/118437540

    sql = 'create table ' + table_name + "_1" + " like " + table_name + ";";
    print(sql)
    cur.execute(sql)
    
    sql = 'create index index_' + colname + " on " + table_name + "_1(" + colname + "(28));"  
    print(sql)
    cur.execute(sql)
    
    sql = 'INSERT into  ' + table_name +  "_1" + " SELECT * from " + table_name + ";"
    print(sql)
    cur.execute(sql)
    
    sql = 'drop table ' + table_name + ";"
    print(sql)
    cur.execute(sql)

    sql = 'rename table ' + table_name + "_1" + " to " + table_name + ";"
    print(sql)
    cur.execute(sql)
    
    exit(0)
    
    # fetchall是获取全部数据
    data = cur.fetchall()
    print(data)

    # 关闭游标
    cur.close()
    # 关闭数据库连接
    db.close()


def create_index(table_name,colname, debug=True):
    db = pymysql.connect(
    host = remote_sql_server , # 服务器地址
    port = 3306, # 端口
    user = "root" , # 用户名
    passwd = password , # 密码
    db = 'stock')
    # 创建游标
    cur = db.cursor()

    sql = 'create index ' + colname +"_index" + " on " + table_name+"("+ colname + ");"
    print(sql)
    # 执行创建sql语句
    cur.execute(sql)

    # fetchall是获取全部数据
    data = cur.fetchall()
    print(data)

    # 关闭游标
    cur.close()
    # 关闭数据库连接
    db.close()

#return dateframe of the stockid, pass in :stock id, only query from internet when local db doesn't have it.
def stock_individual_info_em(stockid, debug = True):
    sql = 'select * from stock_individual_info where stock_id=' + stockid;
    table_exist, rows = get_dateframe_result_ex(sql, debug = True)
    if rows.empty:
        #
        print("doesn't exist in mysql for stock_id:" + stockid);
        
        #query from internet:
        try:
            stock_individual_info_em_df = ak.stock_individual_info_em(symbol=stockid)
        except Exception as e:
            print("Got exception when call ak interface ak.stock_individual_info_em" + stockid )
            print(f"{e}");
            traceback.print_exc()
            return pd.DataFrame();

        #create table for the first time.
        df1_indexed = stock_individual_info_em_df.set_index(["item"])
        df1_to_insert=df1_indexed.T
        df1_to_insert.rename(columns={"股票代码": "stock_id"},  inplace = True, errors="raise")
        if debug:
            print(df1_to_insert) 

        
        try:
            df1_to_insert.to_sql("stock_individual_info", engine,if_exists='append', index=False)
        except Exception as e:
            print("Got exception when insert into db stock_individual_info" + stockid )
            print(f"{e}");
            traceback.print_exc()
            return pd.DataFrame();

        #create index for the new table:
        if not table_exist:
            #use this to also set length on the index fieid, and also create index:
            change_col_to_varchar("stock_individual_info", "stock_id");
            #don't use this
            #create_index("stock_individual_info", "share_id");

        time.sleep(5);#sleep to avoid rejection from server.
        return df1_to_insert;
    return rows;
    

#   item                value
#0   总市值  337468917463.220032
#1  流通市值      337466070320.25
#2    行业                   银行
#3  上市时间             19910403
#4  股票代码               000001
#5  股票简称                 平安银行
#6   总股本        19405918198.0
#7   流通股        19405754475.0


import pymysql
import time

def drop_table(table_name, debug=True):
    db = pymysql.connect(
    host = remote_sql_server , # 服务器地址
    port = 3306, # 端口
    user = "root" , # 用户名
    passwd = password , # 密码
    db = 'stock')
    # 创建游标
    cur = db.cursor()

    # 执行创建sql语句
    cur.execute('drop table if exists '+table_name)

    print("drop table:" + table_name)
    # 查看当前数据库所有变信息
    cur.execute('show tables')

    # fetchall是获取全部数据
    data = cur.fetchall()
    print(data)

    # 关闭游标
    cur.close()
    # 关闭数据库连接
    db.close()
    

import akshare as ak

def is_variable_defined(var_name):
    # 获取当前作用域的变量字典
    vars_dict = globals() if var_name in globals() else locals()

    # 判断变量名是否在变量字典中
    if var_name in vars_dict:
        return True
    else:
        return False

    


In [30]:
#experiment:

try:
    drop_table("stock_individual_info")
    #drop_table("stock_individual_info_1")
except pymysql.err.MySQLError as _error:
    raise _error
print("end drop table")


df1 = stock_individual_info_em("600036")
print(df1)

drop table:stock_individual_info
(('conv_bond_details',), ('conv_bond_details2',), ('conv_bond_his',), ('conv_bond_his_stage1',), ('convertible_bond',), ('stock_conv_bond_value_analysis',), ('test_table',))
end drop table
sql exception when executing:select * from stock_individual_info where stock_id=600036
doesn't exist in mysql for stock_id:600036
item                总市值             流通市值  行业      上市时间     最新 stock_id  股票简称  \
value  1090758322243.25  892201846554.25  银行  20020409  43.25   600036  招商银行   

item             总股本            流通股  
value  25219845601.0  20628944429.0  
create table stock_individual_info_1 like stock_individual_info;
create index index_stock_id on stock_individual_info_1(stock_id(28));
INSERT into  stock_individual_info_1 SELECT * from stock_individual_info;
drop table stock_individual_info;
rename table stock_individual_info_1 to stock_individual_info;
[]
item                总市值             流通市值  行业      上市时间     最新 stock_id  股票简称  \
value  1090758322243.2

In [2]:
#experiment:
#get from mysql for the already saved item:
df1 = stock_individual_info_em("600036")
print("\ngot df:\n")
print(df1)

(('总市值', 5, None, 22, 22, 31, True), ('流通市值', 5, None, 22, 22, 31, True), ('行业', 252, None, 262140, 262140, 0, True), ('上市时间', 8, None, 20, 20, 0, True), ('最新', 5, None, 22, 22, 31, True), ('stock_id', 252, None, 262140, 262140, 0, True), ('股票简称', 252, None, 262140, 262140, 0, True), ('总股本', 5, None, 22, 22, 31, True), ('流通股', 5, None, 22, 22, 31, True))
            总市值          流通市值  行业      上市时间     最新 stock_id  股票简称  \
0  1.090758e+12  8.922018e+11  银行  20020409  43.25   600036  招商银行   

            总股本           流通股  
0  2.521985e+10  2.062894e+10

got df:

            总市值          流通市值  行业      上市时间     最新 stock_id  股票简称  \
0  1.090758e+12  8.922018e+11  银行  20020409  43.25   600036  招商银行   

            总股本           流通股  
0  2.521985e+10  2.062894e+10  


In [10]:
df1_indexed = df1_raw.set_index(["item"])
print(df1_indexed.T)
len(df1_indexed.T)

item                总市值             流通市值  行业      上市时间     最新    股票代码  股票简称  \
value  1090758322243.25  892201846554.25  银行  20020409  43.25  600036  招商银行   

item             总股本            流通股  
value  25219845601.0  20628944429.0  


1

In [13]:
df1_to_insert=df1_indexed.T
df1_to_insert.rename(columns={"股票代码": "share_id"},  inplace = True, errors="raise")
print(df1_to_insert) 

item                总市值             流通市值  行业      上市时间     最新 share_id  股票简称  \
value  1090758322243.25  892201846554.25  银行  20020409  43.25   600036  招商银行   

item             总股本            流通股  
value  25219845601.0  20628944429.0  


In [5]:
print(df1)

   item             value
0   总市值  1090758322243.25
1  流通市值   892201846554.25
2    行业                银行
3  上市时间          20020409
4    最新             43.25
5  股票代码            600036
6  股票简称              招商银行
7   总股本     25219845601.0
8   流通股     20628944429.0


In [12]:
#!pip3 install ipywidgets

In [24]:
print(df1)
print(df1.loc["value","总市值"])
df1.index.values[0]

item                总市值             流通市值  行业      上市时间     最新 stock_id  股票简称  \
value  1090758322243.25  892201846554.25  银行  20020409  43.25   600036  招商银行   

item             总股本            流通股  
value  25219845601.0  20628944429.0  
1090758322243.25


'value'

In [28]:
print(df1)
print(df1.loc[df1.index.values[0],"总市值"])
#df1.index.values[0]

            总市值          流通市值  行业      上市时间     最新 stock_id  股票简称  \
0  1.090758e+12  8.922018e+11  银行  20020409  43.25   600036  招商银行   

            总股本           流通股  
0  2.521985e+10  2.062894e+10  
1090758322243.25


In [14]:

def getStockVolume(stockid, match_stock_list):
  stock_individual_info_em_df = stock_individual_info_em(stockid)

  if stock_individual_info_em_df.empty:
    print("got empty df from stock_individual_info_em, could be url request error");
    return;
  index_value_for_the_row = stock_individual_info_em_df.index.values[0]
  volume = stock_individual_info_em_df.loc[index_value_for_the_row,"总市值"];
  #print("getting"+volume)
  #市值小于80 亿，新增热门股票
  if int(volume) < 10000000000:
    print(stockid);
    print(stock_individual_info_em_df.loc[index_value_for_the_row,"stock_id"]);
    match_stock_list.append([stockid, stock_individual_info_em_df.loc[index_value_for_the_row,"股票简称"]])


df1 = stock_individual_info_em("600036")
print("\ngot df:\n")
print(df1)

#0 row, column by index name:
print(df1.loc[0,"总市值"])
match_stock_list=[]
getStockVolume('000002', match_stock_list)
print("returned list:" + str(match_stock_list));

(('总市值', 5, None, 22, 22, 31, True), ('流通市值', 5, None, 22, 22, 31, True), ('行业', 252, None, 262140, 262140, 0, True), ('上市时间', 8, None, 20, 20, 0, True), ('最新', 5, None, 22, 22, 31, True), ('stock_id', 252, None, 262140, 262140, 0, True), ('股票简称', 252, None, 262140, 262140, 0, True), ('总股本', 5, None, 22, 22, 31, True), ('流通股', 5, None, 22, 22, 31, True))
            总市值          流通市值  行业      上市时间     最新 stock_id  股票简称  \
0  1.090758e+12  8.922018e+11  银行  20020409  43.25   600036  招商银行   

            总股本           流通股  
0  2.521985e+10  2.062894e+10

got df:

            总市值          流通市值  行业      上市时间     最新 stock_id  股票简称  \
0  1.090758e+12  8.922018e+11  银行  20020409  43.25   600036  招商银行   

            总股本           流通股  
0  2.521985e+10  2.062894e+10  
1090758322243.25
(('总市值', 5, None, 22, 22, 31, True), ('流通市值', 5, None, 22, 22, 31, True), ('行业', 252, None, 262140, 262140, 0, True), ('上市时间', 8, None, 20, 20, 0, True), ('最新', 5, None, 22, 22, 31, True), ('stock_id', 252, None,

In [7]:
#切换ip

import socket

def get_local_ip():
    try:
        # 创建一个socket对象
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        # 不需要实际连接，只是用来获取IP地址
        s.connect(('8.8.8.8', 80))
        local_ip = s.getsockname()[0]
    finally:
        s.close()
    return local_ip

print("本地IP地址:", get_local_ip())


本地IP地址: 143.198.231.133


In [25]:
#mutlple ip selection:
!pip3 install PySocks

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [35]:
import psutil
import re
info = psutil.net_if_addrs()
def get_local_ips():
    """获取本机所有ip"""
    local_ips = []
    info = psutil.net_if_addrs()
    for k, v in info.items():
        if "eth" in k:
            for item in v:
                if item[0] == 2:
                    local_ips.append(item[1])
    return local_ips
def getNetiAddrInfo():
    """获取IP地址
    代码来源: https://www.programcreek.com/python/example/88702/psutil.net_if_addrs
    """
    neti_list = []
    ipv4_list = []
    ipv6_list = []
    # id_neti_list = []
    # result_list = []
    neti_dict = psutil.net_if_addrs()
    for neti in neti_dict:
        neti_list.append(neti)
        # id_neti_list.append('NETI-{}-{}'.format(self.os_id, neti))
        snic_list = neti_dict[neti]
        for snic in snic_list:
            if snic.family.name == 'AF_INET':
                ipv4_list.append(snic.address)
            elif snic.family.name == 'AF_INET6':
                ipv6_list.append(re.sub('%.*$', '', snic.address))
    # result = [','.join(neti_list), ','.join(ipv4_list + ipv6_list), ','.join(id_neti_list)]
    return list(set(ipv4_list))
# # 去掉原始ip
local_ips = getNetiAddrInfo()
print(local_ips)


import random
import psutil
import requests
from requests_toolbelt import SourceAddressAdapter


def adapter_requests():
    """随机绑定一个本机ip"""
    session = requests.session()
    bind_address = "10.48.0.5"
    print("请求ip：", bind_address)
    new_source = SourceAddressAdapter(bind_address)
    session.mount('http://', new_source)
    session.mount('https://', new_source)

    url = "http://httpbin.org/get"
    response = session.get(url=url)
    origin = response.json()["origin"]
    print("检测到ip：", origin)

adapter_requests();

#test ip request:

import socks
import socket

# 全局代理
#socks.set_default_proxy(socks.HTTP, "139.159.118.14", 5678)
socks.setdefaultproxy()
socket.socket = socks.socksocket

#os.environ['HTTP_PROXY'] = '43.198.248.18:475'
#os.environ['HTTPS_PROXY'] = '125.87.95.190:8089'

#os.environ['HTTP_PROXY'] = ''
#os.environ['HTTPS_PROXY'] = ''
resp = requests.get('http://httpbin.org/ip')
#resp = requests.get('https://ip.900cha.com/')
resp.encoding = 'utf8'
import json
#print(json.loads(resp.text))  # 应该显示代理服务器的 IP
print(str(resp.json()))


['143.198.231.133', '10.48.0.5', '172.17.0.1', '10.124.0.2', '127.0.0.1']
请求ip： 10.48.0.5
检测到ip： 209.38.172.88
{'origin': '143.198.231.133'}


In [4]:
#全局替换：
#refer to https://blog.csdn.net/HideInTime/article/details/105785011
import socket
import requests
#import urllib, urllib2

def is_variable_defined(var_name):
    return var_name in globals() or var_name in locals()

if is_variable_defined("_create_socket"):
  import errrrrr;

_create_socket = None;

SOURCE_ADDRESS = ("10.48.0.5", 0)

#SOURCE_ADDRESS = ("172.28.153.121", 0)

#SOURCE_ADDRESS = ("172.16.30.41", 0)

def create_connection(*args, **kwargs):
  in_args = False
  if len(args) >=3:
    args = list(args)
    args[2] = SOURCE_ADDRESS
    args = tuple(args)
    in_args = True
  if not in_args:
    kwargs["source_address"] = SOURCE_ADDRESS
  print("args", args)
  print("kwargs", str(kwargs))
  return _create_socket(*args, **kwargs)

#socket.create_connection = create_connection

import urllib3.connection


if None == _create_socket:
    print("replaced urlib3's connection");
    _create_socket = urllib3.connection.connection.create_connection
    urllib3.connection.connection.create_connection = create_connection



#test 
resp = requests.get('http://httpbin.org/ip')
#resp = requests.get('https://ip.900cha.com/')
resp.encoding = 'utf8'
import json
#print(json.loads(resp.text))  # 应该显示代理服务器的 IP
print(str(resp.json()))




ModuleNotFoundError: No module named 'errrrrr'

In [5]:
#test 
resp = requests.get('http://httpbin.org/ip')
#resp = requests.get('https://ip.900cha.com/')
resp.encoding = 'utf8'
import json
#print(json.loads(resp.text))  # 应该显示代理服务器的 IP
print(str(resp.json()))


args (('httpbin.org', 80), None)
kwargs {'source_address': ('10.48.0.5', 0), 'socket_options': [(6, 1, 1)]}
{'origin': '209.38.172.88'}


In [ ]:
# do 

#市值小于100 亿，新增热门股票

#选取本周新增热门股， 盘子不大的股票


def getStockVolume(stockid, match_stock_list):
  stock_individual_info_em_df = stock_individual_info_em(stockid)

  if stock_individual_info_em_df.empty:
      print("skipped processing stock_id:"+ stockid);
      return False;
  index_value_for_the_row = stock_individual_info_em_df.index.values[0]
  volume = stock_individual_info_em_df.loc[index_value_for_the_row,"总市值"];
  #print("getting"+volume)
  #市值小于80 亿，新增热门股票
  if int(volume) < 10000000000:
    print(stockid);
    print(stock_individual_info_em_df.loc[index_value_for_the_row,"stock_id"]);
    match_stock_list.append([stockid, stock_individual_info_em_df.loc[index_value_for_the_row,"股票简称"]])
  return True;

if not is_variable_defined('stock_hot_follow_xq_df'):
    stock_hot_follow_xq_df = ak.stock_hot_follow_xq(symbol="本周新增")
print(stock_hot_follow_xq_df)


keys = stock_hot_follow_xq_df.iloc[:, 0].tolist()
match_stock_list=[]
for stockid in keys:
    stockid = stockid[2:]
    #print(stockid)
    getStockVolume(stockid, match_stock_list)

    
print("final list:")
print(match_stock_list)




#以上数据再次过滤
#机构评级
stock_profit_forecast_em_df = ak.stock_profit_forecast_em()
print(stock_profit_forecast_em_df)

print(stock_profit_forecast_em_df.loc[:,'机构投资评级(近六个月)-买入'])

df_research = stock_profit_forecast_em_df.loc[:,['名称','代码', '研报数','机构投资评级(近六个月)-买入']]
print("df_research:");
print(df_research) 



print(match_stock_list)
match_stock_list2 = [i[0] for i in match_stock_list];
print(match_stock_list2)
df_research['sortby'] = df_research['研报数'] + df_research['机构投资评级(近六个月)-买入']
df_research.sort_values(by=['sortby'])
print("df_research: after sort:");
print(df_research)





codelist_sorted = df_research.loc[:, '代码'].values
print("codelist_sorted:");
print(codelist_sorted)
for i in codelist_sorted:
  if i in match_stock_list2:
    stockid = i
    #print(stockid + "is in list")
    #print("stock id:"+stockid)
    df_sel = df_research[df_research['代码'] == stockid] 
    if not df_sel.empty:
        print("stock id:"+stockid)
        print(df_sel)

args (('xueqiu.com', 443), None)
kwargs {'source_address': ('10.48.0.5', 0), 'socket_options': [(6, 1, 1)]}


  0%|          | 0/28 [00:00<?, ?it/s]

args (('xueqiu.com', 443), None)
kwargs {'source_address': ('10.48.0.5', 0), 'socket_options': [(6, 1, 1)]}
args (('xueqiu.com', 443), None)
kwargs {'source_address': ('10.48.0.5', 0), 'socket_options': [(6, 1, 1)]}
args (('xueqiu.com', 443), None)
kwargs {'source_address': ('10.48.0.5', 0), 'socket_options': [(6, 1, 1)]}
args (('xueqiu.com', 443), None)
kwargs {'source_address': ('10.48.0.5', 0), 'socket_options': [(6, 1, 1)]}
args (('xueqiu.com', 443), None)
kwargs {'source_address': ('10.48.0.5', 0), 'socket_options': [(6, 1, 1)]}
args (('xueqiu.com', 443), None)
kwargs {'source_address': ('10.48.0.5', 0), 'socket_options': [(6, 1, 1)]}
args (('xueqiu.com', 443), None)
kwargs {'source_address': ('10.48.0.5', 0), 'socket_options': [(6, 1, 1)]}
args (('xueqiu.com', 443), None)
kwargs {'source_address': ('10.48.0.5', 0), 'socket_options': [(6, 1, 1)]}
args (('xueqiu.com', 443), None)
kwargs {'source_address': ('10.48.0.5', 0), 'socket_options': [(6, 1, 1)]}
args (('xueqiu.com', 443), N

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/sqlalchemy/engine/base.py", line 1967, in _exec_single_context
    self.dialect.do_execute(
  File "/usr/local/lib/python3.10/site-packages/sqlalchemy/engine/default.py", line 951, in do_execute
    cursor.execute(statement, parameters)
  File "/usr/local/lib/python3.10/site-packages/mysql/connector/cursor.py", line 416, in execute
    self._connection.cmd_query(
  File "/usr/local/lib/python3.10/site-packages/mysql/connector/opentelemetry/context_propagation.py", line 97, in wrapper
    return method(cnx, *args, **kwargs)
  File "/usr/local/lib/python3.10/site-packages/mysql/connector/_decorating.py", line 89, in handle_cnx_method
    raise err
  File "/usr/local/lib/python3.10/site-packages/mysql/connector/_decorating.py", line 85, in handle_cnx_method
    return cnx_method(cnx, *args, **kwargs)
  File "/usr/local/lib/python3.10/site-packages/mysql/connector/connection.py", line 984, in cmd_query
    r

(('总市值', 5, None, 22, 22, 31, True), ('流通市值', 5, None, 22, 22, 31, True), ('行业', 252, None, 262140, 262140, 0, True), ('上市时间', 8, None, 20, 20, 0, True), ('最新', 5, None, 22, 22, 31, True), ('stock_id', 252, None, 262140, 262140, 0, True), ('股票简称', 252, None, 262140, 262140, 0, True), ('总股本', 5, None, 22, 22, 31, True), ('流通股', 5, None, 22, 22, 31, True))
            总市值          流通市值    行业      上市时间    最新 stock_id  股票简称  \
0  1.209279e+10  1.209279e+10  塑料制品  20000525  12.5   000973  佛塑科技   

           总股本          流通股  
0  967423171.0  967423171.0
(('总市值', 5, None, 22, 22, 31, True), ('流通市值', 5, None, 22, 22, 31, True), ('行业', 252, None, 262140, 262140, 0, True), ('上市时间', 8, None, 20, 20, 0, True), ('最新', 5, None, 22, 22, 31, True), ('stock_id', 252, None, 262140, 262140, 0, True), ('股票简称', 252, None, 262140, 262140, 0, True), ('总股本', 5, None, 22, 22, 31, True), ('流通股', 5, None, 22, 22, 31, True))
            总市值          流通市值    行业      上市时间    最新 stock_id  股票简称  \
0  8.932025e+09  

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/sqlalchemy/engine/base.py", line 1967, in _exec_single_context
    self.dialect.do_execute(
  File "/usr/local/lib/python3.10/site-packages/sqlalchemy/engine/default.py", line 951, in do_execute
    cursor.execute(statement, parameters)
  File "/usr/local/lib/python3.10/site-packages/mysql/connector/cursor.py", line 416, in execute
    self._connection.cmd_query(
  File "/usr/local/lib/python3.10/site-packages/mysql/connector/opentelemetry/context_propagation.py", line 97, in wrapper
    return method(cnx, *args, **kwargs)
  File "/usr/local/lib/python3.10/site-packages/mysql/connector/_decorating.py", line 89, in handle_cnx_method
    raise err
  File "/usr/local/lib/python3.10/site-packages/mysql/connector/_decorating.py", line 85, in handle_cnx_method
    return cnx_method(cnx, *args, **kwargs)
  File "/usr/local/lib/python3.10/site-packages/mysql/connector/connection.py", line 984, in cmd_query
    r

(('总市值', 5, None, 22, 22, 31, True), ('流通市值', 5, None, 22, 22, 31, True), ('行业', 252, None, 262140, 262140, 0, True), ('上市时间', 8, None, 20, 20, 0, True), ('最新', 5, None, 22, 22, 31, True), ('stock_id', 252, None, 262140, 262140, 0, True), ('股票简称', 252, None, 262140, 262140, 0, True), ('总股本', 5, None, 22, 22, 31, True), ('流通股', 5, None, 22, 22, 31, True))
            总市值          流通市值    行业      上市时间   最新 stock_id  股票简称  \
0  8.175174e+09  7.999484e+09  电网设备  20080625  4.1   002256  兆新股份   

            总股本           流通股  
0  1.993945e+09  1.951094e+09
002256
002256
(('总市值', 5, None, 22, 22, 31, True), ('流通市值', 5, None, 22, 22, 31, True), ('行业', 252, None, 262140, 262140, 0, True), ('上市时间', 8, None, 20, 20, 0, True), ('最新', 5, None, 22, 22, 31, True), ('stock_id', 252, None, 262140, 262140, 0, True), ('股票简称', 252, None, 262140, 262140, 0, True), ('总股本', 5, None, 22, 22, 31, True), ('流通股', 5, None, 22, 22, 31, True))
            总市值          流通市值    行业      上市时间     最新 stock_id  股票简称  \
